# Download MODIS and VIIRS Vegetation Indices (NDVI & EVI) with AppEEARS

### Overview

This notebook uses the NASA [AppEEARS API](https://appeears.earthdatacloud.nasa.gov/api/) to download 16-day, 500 m vegetation index composites clipped to a bounding box and reprojected to a regular lat/lon grid (EPSG:4326) as NetCDF4:

| Sensor | Product | Layers | Available from |
|---|---|---|---|
| MODIS Terra | `MOD13A1.061` | `_500m_16_days_NDVI`, `_500m_16_days_EVI`, `_500m_16_days_pixel_reliability` | 2000-02-18 |
| VIIRS S-NPP | `VNP13A1.002` | `500_m_16_days_NDVI`, `500_m_16_days_EVI`, `500_m_16_days_pixel_reliability` | 2012-01-17 |

VIIRS is only requested when the selected time range reaches 2012-01-17 or later.

Vegetation indices are not available through the Giovanni API, so AppEEARS is used instead. It avoids handling raw HDF4 sinusoidal MODIS tiles.

### Requirements

- Earthdata Login account: https://urs.earthdata.nasa.gov/users/new
- [earthaccess](https://earthaccess.readthedocs.io/), [requests](https://requests.readthedocs.io/), [xarray](https://docs.xarray.dev/), [h5netcdf](https://h5netcdf.org/), [matplotlib](https://matplotlib.org/)

### 1. Import Libraries

In [ ]:
import os
import glob
import time
import netrc
from datetime import datetime

import requests
import earthaccess
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline

### 2. Log in to AppEEARS with Earthdata Login Credentials

`earthaccess.login` prompts for Earthdata Login credentials and saves them to `~/.netrc` if it does not exist. The AppEEARS `/login` endpoint uses HTTP Basic auth with those credentials and returns a bearer token valid for about 48 hours.

In [ ]:
API = "https://appeears.earthdatacloud.nasa.gov/api"

# Prompt for Earthdata Login credentials, store in a .netrc file if it does not exist
earthaccess.login(strategy="interactive", persist=True)

# Read the stored credentials and request an AppEEARS token
username, _, password = netrc.netrc().authenticators("urs.earthdata.nasa.gov")
login_response = requests.post(f"{API}/login", auth=(username, password))
login_response.raise_for_status()

headers = {"Authorization": f"Bearer {login_response.json()['token']}"}

### 3. Specify Spatiotemporal Parameters and Products

The bounding box and time range match the MERRA-2 subsetting notebook. 16-day composites overlapping the time range are returned.

`good_reliability` lists the `pixel_reliability` values kept when masking (from the AppEEARS `/quality` endpoint):

- **MODIS** (`MOD13A1.061`): 0 = good, 1 = marginal, 2 = snow/ice, 3 = cloudy.
- **VIIRS** (`VNP13A1.002`): 0 = excellent, 1 = good, 2 = acceptable, 3 = marginal, 4 = pass, 5 = questionable, 6 = poor, 7 = cloud shadow, 8 = snow/ice, 9 = cloud.

In [ ]:
# Define the spatial bounding box (west, south, east, north)
bbox = (-95, 20, -75, 35)

# Specify the start and end dates
start = datetime(2005, 8, 1)
stop = datetime(2005, 9, 30)

# Download directory
data_dir = "../../data/NDVI_Subsets"

PRODUCTS = {
    "MODIS": {
        "product": "MOD13A1.061",
        "prefix": "_500m_16_days_",
        "first_date": datetime(2000, 2, 18),
        "good_reliability": [0, 1],
    },
    "VIIRS": {
        "product": "VNP13A1.002",
        "prefix": "500_m_16_days_",
        "first_date": datetime(2012, 1, 17),
        "good_reliability": [0, 1, 2],
    },
}
LAYERS = ["NDVI", "EVI", "pixel_reliability"]

# Only request products that have data in the selected time range
active_products = {name: p for name, p in PRODUCTS.items() if stop >= p["first_date"]}
print("Requesting:", {name: p["product"] for name, p in active_products.items()})

### 4. Submit the AppEEARS Area Request

The bounding box is converted to a GeoJSON polygon, and the output is requested as NetCDF4 in a geographic (lat/lon) projection.

In [ ]:
west, south, east, north = bbox
geo = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[west, south], [east, south], [east, north], [west, north], [west, south]]],
        },
    }],
}

task = {
    "task_type": "area",
    "task_name": f"vegetation_indices_{start:%Y%m%d}_{stop:%Y%m%d}",
    "params": {
        "dates": [{"startDate": f"{start:%m-%d-%Y}", "endDate": f"{stop:%m-%d-%Y}"}],
        "layers": [
            {"product": p["product"], "layer": p["prefix"] + layer}
            for p in active_products.values()
            for layer in LAYERS
        ],
        "geo": geo,
        "output": {"format": {"type": "netcdf4"}, "projection": "geographic"},
    },
}

task_response = requests.post(f"{API}/task", json=task, headers=headers)
task_response.raise_for_status()
task_id = task_response.json()["task_id"]
print("Task ID:", task_id)

### 5. Wait for the Request to Finish

AppEEARS processes requests asynchronously. Large areas or long time ranges can take from minutes to hours; the task status is checked every 30 seconds.

In [ ]:
while True:
    status = requests.get(f"{API}/task/{task_id}", headers=headers).json()["status"]
    print(f"{datetime.now():%H:%M:%S} status: {status}")
    if status == "done":
        break
    if status == "error":
        raise RuntimeError(f"AppEEARS task {task_id} failed")
    time.sleep(30)

### 6. Download the NetCDF Files

The bundle also contains metadata files (CSV/XML/JSON); only the NetCDF files are downloaded here. Download links redirect to temporary S3 URLs.

In [ ]:
os.makedirs(data_dir, exist_ok=True)

bundle = requests.get(f"{API}/bundle/{task_id}", headers=headers).json()

filelist = []
for f in bundle["files"]:
    if not f["file_name"].endswith(".nc"):
        continue
    path = os.path.join(data_dir, os.path.basename(f["file_name"]))
    with requests.get(f"{API}/bundle/{task_id}/{f['file_id']}", headers=headers, stream=True) as r:
        r.raise_for_status()
        with open(path, "wb") as out:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                out.write(chunk)
    filelist.append(path)

print("\n".join(filelist))

### 7. Open, Mask by Pixel Reliability, and Save

Scale factors and fill values are applied automatically by Xarray. Pixels whose `pixel_reliability` is not in `good_reliability` are set to NaN.

In [ ]:
def get_layer(ds: xr.Dataset, suffix: str) -> xr.DataArray:
    """Return the data variable whose name ends with the given suffix."""
    matches = [v for v in ds.data_vars if str(v).endswith(suffix)]
    if len(matches) != 1:
        raise KeyError(f"Expected one variable ending in {suffix!r}, found {matches} in {list(ds.data_vars)}")
    return ds[matches[0]]


vegetation = {}
for name, p in active_products.items():
    paths = glob.glob(os.path.join(data_dir, f"{p['product']}*.nc"))
    ds = xr.open_mfdataset(paths, engine="h5netcdf")

    good = get_layer(ds, "_pixel_reliability").isin(p["good_reliability"])
    vegetation[name] = xr.Dataset({
        "NDVI": get_layer(ds, "_NDVI").where(good),
        "EVI": get_layer(ds, "_EVI").where(good),
    })
    print(name, vegetation[name])

    # Save the masked dataset to a NetCDF4 file
    vegetation[name].to_netcdf(os.path.join(data_dir, f"{name}_NDVI_EVI_masked.nc4"), engine="h5netcdf")

### 8. Plot NDVI for the First Composite

In [ ]:
fig, axes = plt.subplots(1, len(vegetation), figsize=(7 * len(vegetation), 5), squeeze=False)
for ax, (name, ds) in zip(axes[0], vegetation.items()):
    first = ds["NDVI"].isel(time=0)
    first.plot(ax=ax, cmap="YlGn", vmin=-0.2, vmax=1.0)
    ax.set_title(f"{name} NDVI, composite starting {str(first.time.values)[:10]}")
plt.tight_layout()

### 9. Plot the Area-Averaged NDVI Time Series

In [ ]:
for name, ds in vegetation.items():
    ds["NDVI"].mean(dim=["lat", "lon"]).plot(marker="o", label=name)
plt.ylabel("NDVI")
plt.title("Area-averaged NDVI")
plt.legend();

### 10. Log Out of AppEEARS

In [ ]:
requests.post(f"{API}/logout", headers=headers)